# 04 - Model Final LightGBM V2 dengan GPU T4 x2

**Mahasiswa:** Rahardian Putra  
**NRP:** 5052231018

Notebook ini meningkatkan submission sebelumnya yang memperoleh **Public Macro F1 0,55165**.

Perubahan utama:

1. mempertahankan `Pipeline` dan `ColumnTransformer` untuk mencegah data leakage;
2. membandingkan fitur `building_id` lama dengan fitur prefix hierarkis;
3. memilih bobot ensemble dan multiplier kelas hanya pada validation internal;
4. mengevaluasi pilihan final satu kali pada holdout yang tidak dipakai untuk tuning;
5. melatih ulang pada seluruh train dan membuat `submission_improved_v2.csv`.

Model yang digunakan adalah **LightGBM gradient boosting**, bukan Neural Network. Metrik resmi adalah Macro F1. Jangan mengirim submission baru jika skor holdout V2 tidak melampaui acuan 0,5483 dari eksperimen sebelumnya.

## 1. Setup dan konfigurasi GPU T4 ×2

Di Kaggle, buka **Settings → Accelerator → GPU T4 x2** sebelum menjalankan notebook. Mode penuh memakai kedua GPU: notebook mencoba backend CUDA multi-GPU terlebih dahulu, kemudian OpenCL dengan dua pelatihan paralel pada GPU 0 dan GPU 1. `FAST_MODE=True` hanya untuk memeriksa alur dan tidak dipakai sebagai submission final.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import joblib
import lightgbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", message="X does not have valid feature names")

SEED = 42
LABELS = [f"Grade {number}" for number in range(1, 6)]
LABEL_TO_CODE = {label: number for number, label in enumerate(LABELS, start=1)}
OLD_HOLDOUT_REFERENCE = 0.5483
OLD_PUBLIC_REFERENCE = 0.55165
FAST_MODE = False
FINAL_SEEDS = [42] if FAST_MODE else [42, 179]


def detect_nvidia_gpus() -> list[str]:
    """Return NVIDIA GPU names exposed to the current Kaggle runtime."""
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        return []
    return [line.strip() for line in result.stdout.splitlines() if line.strip()]


def select_lightgbm_gpu_backend(gpu_count: int) -> tuple[str, dict]:
    """Verify that every T4 is available through LightGBM OpenCL."""
    rng = np.random.default_rng(SEED)
    probe_X = rng.normal(size=(4096, 16)).astype("float32")
    probe_y = rng.integers(0, 5, size=len(probe_X))

    try:
        for device_id in range(gpu_count):
            probe = LGBMClassifier(
                n_estimators=2,
                num_leaves=15,
                max_bin=63,
                device_type="gpu",
                gpu_device_id=device_id,
                gpu_use_dp=False,
                verbosity=-1,
                n_jobs=1,
                random_state=SEED,
            )
            probe.fit(probe_X, probe_y)
    except Exception as exc:
        raise RuntimeError(
            "LightGBM OpenCL tidak dapat memakai seluruh GPU T4. Pastikan "
            "Settings → Accelerator adalah GPU T4 x2, kemudian Restart Session. "
            f"Detail: {type(exc).__name__}: {exc}"
        ) from exc

    return "opencl_dual", {
        "device_type": "gpu",
        "gpu_use_dp": False,
    }


GPU_NAMES = detect_nvidia_gpus()
if len(GPU_NAMES) < 2:
    raise RuntimeError(
        f"Notebook membutuhkan GPU T4 x2, tetapi hanya mendeteksi {len(GPU_NAMES)} GPU: "
        f"{GPU_NAMES or '[tidak ada]'}. Pilih Settings → Accelerator → GPU T4 x2, "
        "lalu Restart Session dan Run All."
    )

GPU_BACKEND, GPU_BASE_PARAMS = select_lightgbm_gpu_backend(len(GPU_NAMES))
CPU_THREADS_PER_MODEL = max(1, (os.cpu_count() or 2) // 2)

pd.set_option("display.max_columns", 80)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| sklearn:", sklearn.__version__, "| lightgbm:", lightgbm.__version__)
print("GPU terdeteksi:", GPU_NAMES)
print("Backend LightGBM:", GPU_BACKEND, "| parameter:", GPU_BASE_PARAMS)


## 2. Muat dan audit data kompetisi

Notebook mencari dataset yang terpasang otomatis pada Kaggle. `building_id` wajib unik, train/test tidak boleh beririsan, dan submission mengikuti urutan `sample_submission.csv`.


In [ ]:
def find_data_dir() -> Path:
    """Return the folder containing the three official competition CSV files."""
    candidates = [
        Path("/kaggle/input/competitions/structural-damage-classification"),
        Path("/kaggle/input/structural-damage-classification"),
        Path.cwd() / "data",
    ]
    for folder in candidates:
        if all((folder / name).exists() for name in ["train.csv", "test.csv", "sample_submission.csv"]):
            return folder
    for train_path in Path("/kaggle/input").glob("**/train.csv"):
        if (train_path.parent / "test.csv").exists() and (train_path.parent / "sample_submission.csv").exists():
            return train_path.parent
    raise FileNotFoundError("Dataset kompetisi tidak ditemukan. Tambahkan competition data sebagai Input.")


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPORT_DIR = OUTPUT_DIR / "reports_v2"
MODEL_DIR = OUTPUT_DIR / "models_v2"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert train.shape == (533465, 17), f"Shape train berubah: {train.shape}"
assert test.shape == (228629, 16), f"Shape test berubah: {test.shape}"
assert set(train.damage_grade.unique()) == set(LABELS)
assert train.building_id.is_unique and test.building_id.is_unique
assert not (set(train.building_id) & set(test.building_id))
assert set(train.columns) - {"damage_grade"} == set(test.columns)
assert list(sample.columns) == ["building_id", "damage_grade"]
assert sample.building_id.equals(test.building_id)

print("Data directory:", DATA_DIR)
print("Train:", train.shape, "| Test:", test.shape)
display(train.head())
display(train.damage_grade.value_counts().sort_index().to_frame("jumlah"))


## 3. Audit struktur `building_id`

Model sebelumnya memakai beberapa potongan dua digit secara terpisah. V2 menambahkan prefix kumulatif sehingga interaksi lokasi tetap utuh, misalnya empat, enam, dan delapan digit pertama. Tabel berikut mengukur kardinalitas dan persentase nilai test yang pernah muncul di train. Ini tidak memakai target test.


In [ ]:
def prefix_audit(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> pd.DataFrame:
    """Summarize building-id prefix cardinality and train-to-test coverage."""
    train_id = train_frame.building_id.astype(str).str.zfill(12)
    test_id = test_frame.building_id.astype(str).str.zfill(12)
    rows = []
    for length in range(2, 11):
        train_prefix = train_id.str[:length]
        test_prefix = test_id.str[:length]
        rows.append({
            "prefix_length": length,
            "train_unique": int(train_prefix.nunique()),
            "test_unique": int(test_prefix.nunique()),
            "test_seen_in_train_pct": round(float(test_prefix.isin(set(train_prefix)).mean() * 100), 3),
            "mean_train_rows_per_group": round(float(len(train_prefix) / train_prefix.nunique()), 2),
        })
    return pd.DataFrame(rows)


prefix_table = prefix_audit(train, test)
display(prefix_table)
prefix_table.to_csv(REPORT_DIR / "building_id_prefix_audit.csv", index=False)


## 4. Feature engineering, ColumnTransformer, dan Pipeline

Terdapat dua profil fitur:

- `segments`: pendekatan submission sebelumnya;
- `hierarchical`: menambah prefix 4, 6, dan 8 digit serta beberapa potongan ID.

Seluruh transformasi tetap berada di dalam Pipeline. Untuk kompatibilitas GPU, kategori berkardinalitas rendah tetap diproses sebagai kategori native LightGBM. ID geografis dan prefix dengan banyak nilai di-encode secara ordinal lalu dianggap numerik, sehingga LightGBM membatasinya menjadi `max_bin=63` dan tidak membuat 919–5.654 categorical bins.


In [ ]:
class BuildingFeatures(BaseEstimator, TransformerMixin):
    """Create deterministic pre-earthquake features for one feature profile."""

    def __init__(self, profile: str = "segments"):
        self.profile = profile

    def fit(self, X: pd.DataFrame, y=None):
        """Store the input schema without learning target-dependent statistics."""
        if self.profile not in {"segments", "hierarchical"}:
            raise ValueError("profile harus 'segments' atau 'hierarchical'.")
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Return engineered features while preserving input row order."""
        X = X.loc[:, self.feature_names_in_].copy()
        building_id = X["building_id"].astype(str).str.zfill(12)
        X = X.drop(columns="building_id")

        for column in ["district_id", "vdcmun_id", "ward_id"]:
            X[column] = X[column].astype("string").fillna("missing").astype(object)

        floors = X["count_floors_pre_eq"].replace(0, np.nan)
        X["height_per_floor"] = X["height_ft_pre_eq"] / floors
        X["estimated_floor_area"] = X["plinth_area_sq_ft"] * floors
        X["age_is_999"] = X["age_building"].eq(999).astype("int8")
        X["log_plinth_area"] = np.log1p(X["plinth_area_sq_ft"].clip(lower=0))

        # Fitur submission sebelumnya.
        for chunk in [2, 3, 4]:
            start = 2 * chunk
            X[f"bid_chunk_{chunk}"] = building_id.str.slice(start, start + 2).astype(object)

        if self.profile == "hierarchical":
            # Prefix menjaga interaksi antarpotongan lokasi yang hilang pada chunk terpisah.
            for length in [4, 6, 8]:
                X[f"bid_prefix_{length}"] = building_id.str[:length].astype(object)
            X["bid_suffix_2"] = building_id.str[-2:].astype(object)

        return X.replace([np.inf, -np.inf], np.nan)


def transformed_schema(X: pd.DataFrame, profile: str) -> tuple[list[str], list[str]]:
    """Return numeric and categorical columns created by a feature profile."""
    transformed = BuildingFeatures(profile).fit_transform(X.head(5))
    numeric = transformed.select_dtypes(include=np.number).columns.tolist()
    categorical = [column for column in transformed.columns if column not in numeric]
    return numeric, categorical


def categorical_cardinalities(
    X: pd.DataFrame,
    profile: str,
    categorical_columns: list[str],
) -> dict[str, int]:
    """Count category levels from training data without using the target."""
    building_id = X["building_id"].astype(str).str.zfill(12)
    counts = {}
    for column in categorical_columns:
        if column in X.columns:
            counts[column] = int(X[column].nunique(dropna=False))
        elif column.startswith("bid_chunk_"):
            chunk = int(column.rsplit("_", 1)[1])
            start = 2 * chunk
            counts[column] = int(
                building_id.str.slice(start, start + 2).nunique(dropna=False)
            )
        elif column.startswith("bid_prefix_"):
            length = int(column.rsplit("_", 1)[1])
            counts[column] = int(
                building_id.str[:length].nunique(dropna=False)
            )
        elif column == "bid_suffix_2":
            counts[column] = int(building_id.str[-2:].nunique(dropna=False))
        else:
            raise KeyError(f"Kardinalitas {column!r} tidak dapat dihitung.")
    return counts


def build_pipeline(
    X: pd.DataFrame,
    profile: str,
    params: dict,
    seed: int = SEED,
    gpu_device_id: int = 0,
) -> tuple[Pipeline, list[int]]:
    """Build a leakage-safe pipeline with GPU-safe categorical bins."""
    numeric, categorical = transformed_schema(X, profile)
    cardinalities = categorical_cardinalities(X, profile, categorical)

    # Kategori besar tidak boleh ditandai sebagai categorical_feature pada GPU.
    low_cardinality = [
        column for column in categorical
        if cardinalities[column] <= 128
    ]
    high_cardinality = [
        column for column in categorical
        if column not in low_cardinality
    ]

    ordinal_steps = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            dtype=np.float32,
        )),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric),
        ("high_cardinality_as_numeric", ordinal_steps, high_cardinality),
        ("low_cardinality_native", ordinal_steps, low_cardinality),
    ], remainder="drop", verbose_feature_names_out=False)

    accelerator_params = dict(GPU_BASE_PARAMS)
    accelerator_params["gpu_device_id"] = gpu_device_id

    estimator = LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        max_bin=63,
        max_cat_threshold=32,
        verbosity=-1,
        n_jobs=CPU_THREADS_PER_MODEL,
        random_state=seed,
        **accelerator_params,
        **params,
    )
    pipeline = Pipeline([
        ("features", BuildingFeatures(profile)),
        ("preprocess", preprocessor),
        ("model", estimator),
    ])

    first_categorical_index = len(numeric) + len(high_cardinality)
    categorical_indices = list(range(
        first_categorical_index,
        first_categorical_index + len(low_cardinality),
    ))
    print(
        f"{profile}: native categorical={low_cardinality}; "
        f"high-cardinality numeric={high_cardinality}"
    )
    return pipeline, categorical_indices


BASE_PARAMS = {
    "n_estimators": 450,
    "learning_rate": 0.08,
    "num_leaves": 63,
    "min_child_samples": 50,
    "colsample_bytree": 0.7,
    "reg_lambda": 10.0,
}

HIERARCHICAL_PARAMS = {
    "n_estimators": 700,
    "learning_rate": 0.05,
    "num_leaves": 255,
    "min_child_samples": 20,
    "colsample_bytree": 0.8,
    "reg_lambda": 10.0,
}

for profile in ["segments", "hierarchical"]:
    numeric, categorical = transformed_schema(train.drop(columns="damage_grade"), profile)
    print(profile, "->", len(numeric), "numeric,", len(categorical), "categorical")


## 5. Pembagian data tanpa kebocoran

Data dibagi menjadi development 80% dan holdout 20%. Development dibagi lagi menjadi fit dan validation. Semua pemilihan fitur, bobot ensemble, dan multiplier kelas dilakukan pada validation internal. Holdout baru dipakai sekali setelah keputusan terkunci.


In [ ]:
X = train.drop(columns="damage_grade")
y = train.damage_grade

X_dev, X_hold, y_dev, y_hold = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.20, stratify=y_dev, random_state=SEED + 1)

if FAST_MODE:
    X_fit, _, y_fit, _ = train_test_split(
        X_fit, y_fit, train_size=60000, stratify=y_fit, random_state=SEED)
    X_valid, _, y_valid, _ = train_test_split(
        X_valid, y_valid, train_size=20000, stratify=y_valid, random_state=SEED)
    X_hold, _, y_hold, _ = train_test_split(
        X_hold, y_hold, train_size=20000, stratify=y_hold, random_state=SEED)

print(f"Fit: {len(X_fit):,} | Validation: {len(X_valid):,} | Holdout: {len(X_hold):,}")


## 6. Baseline ringkas

Baseline dijalankan pada subset agar notebook tetap praktis. Nilainya menjadi pemeriksaan bahwa model baru benar-benar belajar lebih baik daripada tebakan kelas prior dan model linear.


In [ ]:
BASELINE_ROWS = min(80000 if not FAST_MODE else 20000, len(X_fit))
X_base, _, y_base, _ = train_test_split(
    X_fit, y_fit, train_size=BASELINE_ROWS, stratify=y_fit, random_state=SEED)

dummy = DummyClassifier(strategy="prior").fit(X_base[["count_floors_pre_eq"]], y_base)
dummy_f1 = f1_score(y_valid, dummy.predict(X_valid[["count_floors_pre_eq"]]), average="macro")

raw_numeric = ["count_floors_pre_eq", "age_building", "plinth_area_sq_ft", "height_ft_pre_eq"]
raw_categorical = [column for column in X.columns if column not in raw_numeric + ["building_id"]]
linear_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), raw_numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), raw_categorical),
])
logistic = Pipeline([
    ("preprocess", linear_preprocessor),
    ("model", LogisticRegression(max_iter=1000, tol=1e-3, class_weight="balanced", random_state=SEED)),
])
logistic.fit(X_base, y_base)
logistic_f1 = f1_score(y_valid, logistic.predict(X_valid), average="macro")

baseline_table = pd.DataFrame([
    {"model": "Dummy prior", "train_rows": len(X_base), "validation_macro_f1": dummy_f1},
    {"model": "Logistic Regression", "train_rows": len(X_base), "validation_macro_f1": logistic_f1},
])
display(baseline_table)


## 7. Bandingkan model lama dan fitur hierarkis

Kedua model dilatih pada data fit yang sama dan dibandingkan pada validation internal. Probabilitas keduanya disimpan untuk mencari bobot blend dan multiplier kelas.


In [ ]:
def fit_profile(
    X_train,
    y_train,
    profile: str,
    params: dict,
    seed: int = SEED,
    gpu_device_id: int = 0,
):
    """Fit one profile on the selected LightGBM GPU backend."""
    pipeline, categorical_indices = build_pipeline(
        X_train, profile, params, seed, gpu_device_id
    )
    started = time.perf_counter()
    pipeline.fit(
        X_train,
        y_train,
        model__categorical_feature=categorical_indices,
    )
    elapsed = (time.perf_counter() - started) / 60
    print(
        f"{profile} seed={seed} GPU={gpu_device_id} "
        f"selesai dalam {elapsed:.1f} menit"
    )
    return pipeline


def fit_model_specs(X_train, y_train, specs: list[tuple[str, dict, int]]) -> list[Pipeline]:
    """Fit specifications with native multi-GPU or one concurrent job per GPU."""
    if GPU_BACKEND == "cuda_multi":
        return [
            fit_profile(X_train, y_train, profile, params, seed, 0)
            for profile, params, seed in specs
        ]

    def run(index_and_spec):
        index, (profile, params, seed) = index_and_spec
        return fit_profile(
            X_train,
            y_train,
            profile,
            params,
            seed,
            gpu_device_id=index % len(GPU_NAMES),
        )

    with ThreadPoolExecutor(max_workers=len(GPU_NAMES)) as executor:
        return list(executor.map(run, enumerate(specs)))


segment_model, hierarchical_model = fit_model_specs(
    X_fit,
    y_fit,
    [
        ("segments", BASE_PARAMS, SEED),
        ("hierarchical", HIERARCHICAL_PARAMS, SEED),
    ],
)

classes = segment_model.named_steps["model"].classes_
assert np.array_equal(classes, hierarchical_model.named_steps["model"].classes_)

segment_valid_proba = segment_model.predict_proba(X_valid)
hierarchical_valid_proba = hierarchical_model.predict_proba(X_valid)

profile_results = pd.DataFrame([
    {"model": "segments", "validation_macro_f1": f1_score(y_valid, classes[segment_valid_proba.argmax(axis=1)], average="macro")},
    {"model": "hierarchical", "validation_macro_f1": f1_score(y_valid, classes[hierarchical_valid_proba.argmax(axis=1)], average="macro")},
])
display(profile_results)


## 8. Optimasi blend dan multiplier kelas

Macro F1 memberi bobot setara pada setiap kelas. `class_weight="balanced"` membantu saat training, tetapi keputusan argmax masih dapat dikoreksi secara ringan. Pencarian berikut hanya memakai validation internal dan memiliki sedikit parameter agar tidak mudah overfit.


In [ ]:
def predict_with_multipliers(proba: np.ndarray, classes_: np.ndarray,
                             multipliers: np.ndarray) -> np.ndarray:
    """Convert probabilities to labels after applying per-class multipliers."""
    return classes_[(proba * multipliers.reshape(1, -1)).argmax(axis=1)]


def optimize_multipliers(proba: np.ndarray, y_true: pd.Series,
                         classes_: np.ndarray) -> tuple[np.ndarray, float]:
    """Coordinate-search conservative class multipliers on validation predictions."""
    multipliers = np.ones(len(classes_), dtype=float)
    best = f1_score(y_true, predict_with_multipliers(proba, classes_, multipliers), average="macro")
    grid = np.arange(0.80, 1.201, 0.05)
    for _ in range(3):
        changed = False
        for class_index in range(len(classes_)):
            local_best = best
            local_value = multipliers[class_index]
            for value in grid:
                candidate = multipliers.copy()
                candidate[class_index] = value
                score = f1_score(y_true, predict_with_multipliers(proba, classes_, candidate), average="macro")
                if score > local_best + 1e-7:
                    local_best, local_value = score, value
            if local_value != multipliers[class_index]:
                multipliers[class_index] = local_value
                best = local_best
                changed = True
        if not changed:
            break
    # Normalisasi hanya untuk kemudahan membaca; argmax tidak berubah.
    multipliers /= multipliers.mean()
    return multipliers, best


search_rows = []
best_choice = None
for hierarchical_weight in np.arange(0.0, 1.01, 0.10):
    blended = ((1.0 - hierarchical_weight) * segment_valid_proba
               + hierarchical_weight * hierarchical_valid_proba)
    multipliers, score = optimize_multipliers(blended, y_valid, classes)
    row = {
        "hierarchical_weight": round(float(hierarchical_weight), 2),
        "validation_macro_f1": score,
        **{f"multiplier_{label}": float(value) for label, value in zip(classes, multipliers)},
    }
    search_rows.append(row)
    if best_choice is None or score > best_choice[0]:
        best_choice = (score, float(hierarchical_weight), multipliers.copy())

blend_table = pd.DataFrame(search_rows).sort_values("validation_macro_f1", ascending=False)
display(blend_table.head(11))

BEST_VALID_F1, BEST_HIERARCHICAL_WEIGHT, BEST_MULTIPLIERS = best_choice
print(f"Validation terbaik: {BEST_VALID_F1:.5f}")
print(f"Bobot hierarchical: {BEST_HIERARCHICAL_WEIGHT:.2f}")
print("Multiplier:", dict(zip(classes, BEST_MULTIPLIERS.round(4))))


## 9. Evaluasi final satu kali pada holdout

Setelah fitur, bobot blend, dan multiplier terkunci, kedua model dilatih ulang pada seluruh development. Holdout belum pernah dipakai untuk keputusan di atas.


In [ ]:
segment_dev = fit_profile(X_dev, y_dev, "segments", BASE_PARAMS)
hierarchical_dev = fit_profile(X_dev, y_dev, "hierarchical", HIERARCHICAL_PARAMS)

segment_hold_proba = segment_dev.predict_proba(X_hold)
hierarchical_hold_proba = hierarchical_dev.predict_proba(X_hold)
holdout_proba = ((1.0 - BEST_HIERARCHICAL_WEIGHT) * segment_hold_proba
                 + BEST_HIERARCHICAL_WEIGHT * hierarchical_hold_proba)
holdout_prediction = predict_with_multipliers(
    holdout_proba, classes, BEST_MULTIPLIERS)
holdout_f1 = f1_score(y_hold, holdout_prediction, average="macro")

print(f"Holdout Macro F1 V2 : {holdout_f1:.5f}")
print(f"Acuan model lama    : {OLD_HOLDOUT_REFERENCE:.5f}")
print(f"Perubahan           : {holdout_f1 - OLD_HOLDOUT_REFERENCE:+.5f}")
print(classification_report(y_hold, holdout_prediction, labels=LABELS, zero_division=0))

cm = confusion_matrix(y_hold, holdout_prediction, labels=LABELS)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(cmap="Blues", xticks_rotation=30)
plt.title(f"Improved V2 - Holdout Macro F1 {holdout_f1:.4f}")
plt.show()

if holdout_f1 > OLD_HOLDOUT_REFERENCE + 0.002:
    print("REKOMENDASI: kandidat V2 layak dibuat dan dibandingkan di public leaderboard.")
else:
    print("REKOMENDASI: pertahankan submission 0,55165; peningkatan holdout belum meyakinkan.")


## 10. Refit seluruh train dan ensemble dua seed

Submission dibuat hanya setelah evaluasi holdout tercatat. Dua seed mengurangi varians tanpa mengganti jenis model. Probabilitas semua model dirata-ratakan per profil, lalu digabung menggunakan bobot yang dipilih pada validation internal.


In [ ]:
final_specs = [
    ("segments", BASE_PARAMS, seed) for seed in FINAL_SEEDS
] + [
    ("hierarchical", HIERARCHICAL_PARAMS, seed) for seed in FINAL_SEEDS
]

final_models = fit_model_specs(X, y, final_specs)
number_of_seeds = len(FINAL_SEEDS)
segment_final_models = final_models[:number_of_seeds]
hierarchical_final_models = final_models[number_of_seeds:]

segment_test_proba = np.mean(
    [model.predict_proba(test) for model in segment_final_models], axis=0
)
hierarchical_test_proba = np.mean(
    [model.predict_proba(test) for model in hierarchical_final_models], axis=0
)
test_proba = (
    (1.0 - BEST_HIERARCHICAL_WEIGHT) * segment_test_proba
    + BEST_HIERARCHICAL_WEIGHT * hierarchical_test_proba
)
test_labels = predict_with_multipliers(
    test_proba, classes, BEST_MULTIPLIERS
)

prediction_by_id = pd.Series(test_labels, index=test.building_id)
submission = sample.copy()
submission["damage_grade"] = (
    submission.building_id
    .map(prediction_by_id)
    .map(LABEL_TO_CODE)
    .astype("int64")
)

SUBMISSION_PATH = OUTPUT_DIR / (
    "submission_improved_v2_quick.csv"
    if FAST_MODE
    else "submission_improved_v2.csv"
)
submission.to_csv(SUBMISSION_PATH, index=False)

saved = pd.read_csv(SUBMISSION_PATH)
assert saved.shape == (228629, 2)
assert list(saved.columns) == ["building_id", "damage_grade"]
assert saved.building_id.equals(sample.building_id)
assert saved.damage_grade.dtype.kind in "iu"
assert set(saved.damage_grade.unique()) <= {1, 2, 3, 4, 5}
assert not saved.isna().any().any()

print("Submission siap:", SUBMISSION_PATH)
print("Shape:", saved.shape)
display(saved.head())
display(saved.damage_grade.value_counts().sort_index().to_frame("jumlah_prediksi"))


## 11. Simpan model dan laporan

Artefak menyimpan kedua kelompok model, bobot blend, multiplier, dan versi library. Public score tetap kosong sampai file benar-benar dikirim ke Kaggle.


In [ ]:
artifact = {
    "segment_models": segment_final_models,
    "hierarchical_models": hierarchical_final_models,
    "classes": classes,
    "hierarchical_weight": BEST_HIERARCHICAL_WEIGHT,
    "class_multipliers": BEST_MULTIPLIERS,
    "base_params": BASE_PARAMS,
    "hierarchical_params": HIERARCHICAL_PARAMS,
    "gpu_backend": GPU_BACKEND,
    "gpu_names": GPU_NAMES,
}
MODEL_PATH = MODEL_DIR / "lightgbm_blend_v2.joblib"
joblib.dump(artifact, MODEL_PATH)

# Verifikasi round trip pada sampel test.
restored = joblib.load(MODEL_PATH)
restored_segment = np.mean([model.predict_proba(test.head(20)) for model in restored["segment_models"]], axis=0)
restored_hierarchical = np.mean([model.predict_proba(test.head(20)) for model in restored["hierarchical_models"]], axis=0)
restored_proba = ((1.0 - restored["hierarchical_weight"]) * restored_segment
                  + restored["hierarchical_weight"] * restored_hierarchical)
restored_labels = predict_with_multipliers(
    restored_proba, restored["classes"], restored["class_multipliers"])
assert np.array_equal(restored_labels, test_labels[:20])

report = classification_report(y_hold, holdout_prediction, labels=LABELS,
                               output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(REPORT_DIR / "classification_report_v2.csv")
blend_table.to_csv(REPORT_DIR / "blend_search_v2.csv", index=False)
profile_results.to_csv(REPORT_DIR / "profile_comparison_v2.csv", index=False)

summary = {
    "competition": "structural-damage-classification",
    "metric": "macro_f1",
    "previous_public_score": OLD_PUBLIC_REFERENCE,
    "previous_holdout_score": OLD_HOLDOUT_REFERENCE,
    "validation_macro_f1": round(float(BEST_VALID_F1), 6),
    "holdout_macro_f1": round(float(holdout_f1), 6),
    "holdout_improvement": round(float(holdout_f1 - OLD_HOLDOUT_REFERENCE), 6),
    "hierarchical_weight": BEST_HIERARCHICAL_WEIGHT,
    "class_multipliers": {label: round(float(value), 6) for label, value in zip(classes, BEST_MULTIPLIERS)},
    "seeds": FINAL_SEEDS,
    "gpu_backend": GPU_BACKEND,
    "gpu_names": GPU_NAMES,
    "prediction_distribution": {str(key): int(value) for key, value in saved.damage_grade.value_counts().sort_index().items()},
    "submission_file": SUBMISSION_PATH.name,
    "public_score": None,
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "lightgbm": lightgbm.__version__,
    },
}
(REPORT_DIR / "results_v2.json").write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("Model round-trip terverifikasi:", MODEL_PATH)


## 12. Keputusan submission

Gunakan aturan berikut:

- jika holdout V2 naik minimal sekitar 0,002 dari 0,5483 dan distribusi prediksi masuk akal, kirim `submission_improved_v2.csv` sebagai submission kedua;
- jika tidak naik, pertahankan submission lama dengan public score 0,55165;
- jangan memilih model hanya dari satu public score karena leaderboard public memakai sekitar 30% test, sedangkan hasil final memakai 70% sisanya;
- simpan kedua submission yang berhasil sebagai kandidat final bila Kaggle mengizinkan dua pilihan.

Notebook ini tidak otomatis menekan tombol Submit. Catat public score baru setelah Kaggle selesai mengevaluasi file.
